In [1]:
# LE BUT : avoir par dispositif DEPHY_FERME un seul et unique numéro DEPHY en DETAILLE, passer les autres en ALLEGE. Nettoyaer les code_dephy qui sont pas bons et trouver les numéros DEPHY manquants !

In [2]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import re
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

In [3]:
path = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
donnees = dict(
    sdc = pd.read_csv(path + 'sdc.csv'),
    dispositif = pd.read_csv(path + 'dispositif.csv'),
    domaine = pd.read_csv(path + 'domaine.csv'),
    intv_real = pd.read_csv(path + 'intervention_realise_agrege.csv', usecols=['id','sdc_id']),
    intv_synth = pd.read_csv(path + 'intervention_synthetise_agrege.csv', usecols=['id','sdc_id'])
)

In [4]:
sdc = donnees['sdc'][['id','code','nom','code_dephy','reseaux_ir','dispositif_id','modalite_suivi_dephy']].rename(columns={
    'id':'sdc_id',
    'code':'sdc_code',
    'nom':'sdc_nom'})
dispositif = donnees['dispositif'][['id','code','type','domaine_id']].rename(columns={
    'id':'dispositif_id',
    'code':'dispositif_code',
    'type':'dispositif_type'})
domaine = donnees['domaine'][['id','nom','code','commune_id','campagne']].rename(columns={
    'id':'domaine_id',
    'code':'domaine_code',
    'nom' : 'domaine_nom'})
intv = pd.concat([donnees['intv_real'], donnees['intv_synth']], ignore_index=True)

df = sdc.merge(dispositif, on='dispositif_id', how='left')
df = df.merge(domaine, on='domaine_id', how='left')

In [5]:
dfferme = df.loc[(df['dispositif_type']=='DEPHY_FERME')].drop(columns=['dispositif_type'])
del(df)

In [6]:
# pattern_exact = r'^[A-Z]{3}(?:[0-9]|X)\d{4}$'
# pattern_approx = r'[A-Z]{3}(?:[0-9]|X)\d{4}(?!\d)'
# pattern_court = r'[A-Z]{3}(?:[0-9]|X)\d{3}(?!\d)'
# pattern_long = r'[A-Z]{3}(?:[0-9]|X)\d{5}(?!\d)'

In [7]:
# # 1. Nettoyage des numéros DEPHY
# dfferme['code_dephy'] = dfferme['code_dephy'].astype(str).\
#     str.replace('PPZ', '').\
#         str.replace('_', '').\
#             str.replace('-', '').\
#                 str.replace(' ', '').\
#                     str.replace('\t', '').\
#                         str.upper().\
#     where(dfferme['code_dephy'].notna(), None)

# dfferme['code_dephy'] = np.where(dfferme['code_dephy']=='', None, dfferme['code_dephy'])

In [8]:
def verifier_statut(row, df_attendu):
    codes = row['code_dephy']
    annee = row['campagne']

    if codes is None or codes == []:
        return None
    if not isinstance(codes, str):
        codes = str(codes)

    if codes in df_attendu['codes_SdC'].values:
        ligne_attendue = df_attendu[df_attendu['codes_SdC'] == codes].iloc[0]
        valeur_annee = ligne_attendue.get(str(annee), None)

        if valeur_annee == 'Pas de donnees attendues':
            return 'NON-attendue'
        elif valeur_annee is None:
            return 'NON-attendue' 
        else:
            return 'ok'
    else:
        return 'NON-reconnue'

In [9]:
data_attendues = pd.read_csv('/home/administrateur/Bureau/Datagrosyst/catalogue_script_agrosyst/02_outils/data/external_data/' + 'BDD_donnees_attendues_CAN.csv')

dfferme['statut_admin'] = dfferme.apply(lambda row: verifier_statut(row, data_attendues), axis=1)

In [10]:
dfferme

,sdc_id,sdc_code,sdc_nom,code_dephy,reseaux_ir,dispositif_id,modalite_suivi_dephy,dispositif_code,domaine_id,domaine_nom,domaine_code,commune_id,campagne,statut_admin
0,fr.inra.agrosyst.api.entities.GrowingSystem_e8...,243d2f3e-9049-468b-b6d3-beda25de96cf,ARN_SDC_2024,VIF56225,IR_GARCIA,fr.inra.agrosyst.api.entities.GrowingPlan_5620...,DETAILLE,17795ad5-9c50-4f03-85c9-553bcf36eef6,fr.inra.agrosyst.api.entities.Domain_4bef1cea-...,Arnaud Seguy,9a98944c-33aa-4af4-935c-8f1a6bbe70a0,fr.inra.agrosyst.api.entities.referential.RefL...,2024,ok
1,fr.inra.agrosyst.api.entities.GrowingSystem_cc...,2bad1503-8b01-451a-86be-e6b4fd262ab7,pas de maïs danion,NaN,IR_BOISSELIER,fr.inra.agrosyst.api.entities.GrowingPlan_c8b2...,ALLEGE,d4962b45-f33b-4418-85c4-c08bd5313d2b,fr.inra.agrosyst.api.entities.Domain_c315b6f0-...,DANION,2ab4db5e-a8a6-4f08-b30f-9503a944a0c5,fr.inra.agrosyst.api.entities.referential.RefL...,2019,NON-reconnue
2,fr.inra.agrosyst.api.entities.GrowingSystem_3c...,7ff9afc1-002b-4022-a097-a0ba61877860,PYF28007_2016_PP,NaN,IR_CHEVALIER,fr.inra.agrosyst.api.entities.GrowingPlan_6e23...,ALLEGE,675f50e7-00d2-471b-875e-1fd3c282660a,fr.inra.agrosyst.api.entities.Domain_60391c2c-...,DOMAINE_ANNE,9bd78e03-ec1a-4767-9984-f30061a340d3,fr.inra.agrosyst.api.entities.referential.RefL...,2016,NON-reconnue
3,fr.inra.agrosyst.api.entities.GrowingSystem_74...,6993cb4a-2e3c-4117-9cb5-cbd6c9056256,EARL LAPEYRE 2015,GCF10835,IR_BAILLET,fr.inra.agrosyst.api.entities.GrowingPlan_5852...,DETAILLE,80c12023-2d5d-4a07-a8d8-1aa4ea3bd4cf,fr.inra.agrosyst.api.entities.Domain_959f60b8-...,EARL LAPEYRE,bdc5c47f-6620-402c-8083-8649f9e890e2,fr.inra.agrosyst.api.entities.referential.RefL...,2015,ok
4,fr.inra.agrosyst.api.entities.GrowingSystem_ca...,cb5d5a53-2966-4cfd-9bab-c0dae24f47e1,V1_MaJ,LGF36822,IR_GENETIER,fr.inra.agrosyst.api.entities.GrowingPlan_caac...,DETAILLE,971fdb65-b1cd-4c6c-9ff6-c817143e5180,fr.inra.agrosyst.api.entities.Domain_2a12a335-...,Verzotti,c767aba1-1d62-4f83-a8fa-6477de8f1720,fr.inra.agrosyst.api.entities.referential.RefL...,2018,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38545,fr.inra.agrosyst.api.entities.GrowingSystem_0f...,a54798f2-6076-4085-8572-b87ca3762a17,Système_LOPEZ F 2014,LGF10091,IR_LICHOU,fr.inra.agrosyst.api.entities.GrowingPlan_6904...,DETAILLE,844bc2c7-257e-46ce-abef-b86f0e79f26d,fr.inra.agrosyst.api.entities.Domain_97cac0e1-...,Ferme_LEG_LOPEZ F,a4ac5156-aff1-4c0d-85ee-e4c7ab678e5b,fr.inra.agrosyst.api.entities.referential.RefL...,2014,ok
38546,fr.inra.agrosyst.api.entities.GrowingSystem_c8...,8cb8c08b-8905-4bc8-bb39-e72e2153e8d9,MGx2/BTH/Tournesol/CH,GCF10703,IR_GARNIER_Dephy Plaine de l'Ain,fr.inra.agrosyst.api.entities.GrowingPlan_39c4...,DETAILLE,4b999241-b985-4427-bc61-9d91f449c8a1,fr.inra.agrosyst.api.entities.Domain_ca6fb1e1-...,Sylvain Guedon,2edae777-6140-4cfc-ba72-57e16a1715aa,fr.inra.agrosyst.api.entities.referential.RefL...,2015,ok
38547,fr.inra.agrosyst.api.entities.GrowingSystem_29...,d0fc8bb8-7850-46c3-b48e-158de23bd32a,Betteraves-Maïs-Blé-Blé en terres profondes,GCF32816,IR_LAUGIER,fr.inra.agrosyst.api.entities.GrowingPlan_baf7...,DETAILLE,52be2640-d464-4979-89dd-30821a8aa278,fr.inra.agrosyst.api.entities.Domain_1fee6ee9-...,Aurélien Cognet,8bdd90a1-d12d-4528-9453-b468dd9b99fe,fr.inra.agrosyst.api.entities.referential.RefL...,2020,ok
38548,fr.inra.agrosyst.api.entities.GrowingSystem_61...,e5fedbbb-cf10-443f-a356-abf18bb1bf58,SDC_ZWICKERT,LGF57132,IR_GIRAUD,fr.inra.agrosyst.api.entities.GrowingPlan_0290...,DETAILLE,6031874b-43f0-4a1b-813e-a799d04e4b18,fr.inra.agrosyst.api.entities.Domain_ca0a8fb8-...,ZWICKERT Marc,4ccae777-e91f-41e9-8a4d-6ed668b03db2,fr.inra.agrosyst.api.entities.referential.RefL...,2022,NON-attendue


In [ ]:
res = dfferme.loc[dfferme['statut_admin'] != 'ok']

print(f"{len(res) / len(dfferme) * 100:.2f} % des numéro DEPHY ne sont pas conformes à ceux attendus les référentiel de la CAN.\nSoit {len(res)} sdc concernés.")

18.76 % des numéro DEPHY ne sont pas conformes à ceux attendus les référentiel de la CAN.
Soit 6245 sdc concernés.


In [ ]:
res_1 = dfferme.groupby(['code_dephy','campagne'])[['code_dephy','campagne']].value_counts().sort_values(ascending=False)

print(f"{len(res_1.loc[res_1 > 1]) / len(res_1) * 100:.2f} % des combos 'numéro DEPHY * campagne' ont plusieurs sdc associés !\nSoit {len(res_1.loc[res_1 > 1])} combos de {res_1.loc[res_1 > 1].sum()} sdc différents.")

2.57 % des combos 'numéro DEPHY * campagne' ont plusieurs sdc associés !
Soit 752 combos de 1710 sdc différents.


In [ ]:
res_2 = dfferme.groupby('dispositif_id')['code_dephy'].apply(lambda x: x.isna().all() | (x == '').all())

print(f"Parmi les dispositifs FERME contenant au moins un sdc, il y a {len(res_2.loc[res_2]) / len(res_2) * 100:.2f} % des dispositif_id qui n'ont aucun numéro DEPHY renseigné.\nSoit {len(res_2.loc[res_2])} dispositif_id concernés.")

Parmi les dispositifs FERME contenant au moins un sdc, il y a 0.96 % des dispositif_id qui n'ont aucun numéro DEPHY renseigné.
Soit 275 dispositif_id concernés.
